### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following: 

1) Tracking agent behaviour with logging, analytics and debugging
2) Transforming prompts, tool selection, and output formatting
3) Adding retries, fallbacks, and early termination logic
4) Applying rate limits, guardrails, and PII Detection

In [ ]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b", configurable_fields=["model"])
model

### Summarisation middleware

Automatically summarise conversation history when approaching token limits, preserving recent messsages while compressing older context. Summarisation is useful for the following:

1) Long running conversations that exceed context window.
2) Multi-turn dialogues with extensive history
3) Applications where preserving full conversation context matters

In [ ]:
from anyio.lowlevel import checkpoint
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Message based summarisation

agent = create_agent(
    model=model,
    checkpointer = InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model=model,
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [ ]:
### Run with thread id

config={"configurable":{"thread_id":"test1", "model": "groq:llama3-8b-8192"}}

In [ ]:
# Alternative test data

questions=[
    "What is 2 + 2? ",
    "What is 10 * 5? ",
    "What is 100/4? ",
    "What is 15 - 7? ",
    "What is 3 * 3? ",
    "What is 4 * 4? ",
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]}, config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

In [ ]:
from langchain_core.tools import tool 
### Based on token size

@tool

def search_hotel(city :str) -> str:
    """Search hotels - returns long response to use more tokens. """
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, 350$/night, spa, pool, gym
    2. City Inn - 4 star, 180$/night, business center
    3. Budget Stay - 3 star, 75$/night, free wifi"""

agent = create_agent(
    model=model,
    tools=[search_hotel],
    checkpointer = InMemorySaver(),
    middleware = [SummarizationMiddleware(
        model='groq:qwen/qwen3-32b',
        trigger=("tokens",550),
        keep=("tokens",200)
        )
    ]
)

config = {"configurable" : {"thread_id": "test-1"}}

# Token counter

def count_tokens(meesages):
    total_chars = sum(len(str(m.content)) for m in meesages)
    return total_chars

In [ ]:
# Run test

cities = ["Paris","London","Tokyo","New York","Dubai","Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages" : [HumanMessage(content=f"Find hotels in {city}")]},
        config = config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

### Based on fraction

In [ ]:
import time

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000

    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response["messages"])

    time.sleep(2)  # wait 2 seconds before next request

In [ ]:
import uuid
import time

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    # Use a unique thread ID to isolate each search
    city_config = {"configurable": {"thread_id": f"thread-{city}-{uuid.uuid4()}"}}
    
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=city_config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    
    # Optional: short sleep to make sure we don't hit Requests Per Minute limits
    time.sleep(1) 


### Human In the Loop MiddleWare
Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

* High-stakes operations requiring human approval (e.g. database writes, financial transactions).
* Compliance workflows where human oversight is mandatory.
* Long-running conversations where human feedback guides the agent.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body:str) -> str:
    """Mock function to send an email"""
    return f"Email sent to {recipient} with subject '{subject}"


In [ ]:
agent=create_agent(
    model="gpt-4o",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,

            }
        )
    ]
)

In [ ]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [ ]:
result

In [ ]:
# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f"✏️ Result: {result['messages'][-1].content}")